# UBridge AI Guide: PyTorch semantic search
Run the cells top to bottom. A CPU runtime is enough; a GPU only makes training faster.

## 1. Get the project

In [ ]:
REPO_URL = "https://github.com/erich-gagui18/UBridge"
BRANCH = "main"

In [ ]:
import glob, os, subprocess
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "UBridge"], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    subprocess.run(["unzip", "-o", "-q", next(iter(uploaded))], check=True)
ml = os.path.dirname(glob.glob("**/ml/evaluate.py", recursive=True)[0])
os.chdir(ml)
print(os.getcwd())

c:\4th Year 1st Sem\CAP102\UBridge\ml\colab\UBridge\ml


In [ ]:
!pip -q install -r requirements-train.txt pytest

## 2. Smoke test
Trains a tiny random model for two epochs. It must pass before you spend time on the real run.

In [ ]:
!python -m pytest tests/test_train_smoke.py -q

## 3. Baseline: the keyword search that is live now

In [ ]:
!python evaluate.py

## 4. Pretrained embedding models, no training

In [ ]:
!python evaluate.py --models intfloat/multilingual-e5-small intfloat/multilingual-e5-base sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

## 5. Fine-tune with PyTorch (optional)
Pick the model that scored best above.

In [ ]:
MODEL = "intfloat/multilingual-e5-small"

In [ ]:
!python make_pairs.py
!python train.py --model $MODEL --out models/ubridge-embedder --epochs 4

In [ ]:
!python evaluate.py --models $MODEL models/ubridge-embedder

## 6. Build the search index
Set BEST to the winner. A fine-tuned model must be published to the Hugging Face Hub (set HUB_ID) so the service can download it.

In [ ]:
BEST = "intfloat/multilingual-e5-small"
HUB_ID = ""

In [ ]:
if BEST.startswith("models/") and not HUB_ID:
    raise SystemExit("Set HUB_ID (for example your-username/ubridge-embedder) to publish the fine-tuned model.")
if HUB_ID:
    from huggingface_hub import notebook_login
    from sentence_transformers import SentenceTransformer
    notebook_login()
    SentenceTransformer(BEST).push_to_hub(HUB_ID)
record = f"--record-as {HUB_ID}" if HUB_ID else ""
!python build_index.py --model $BEST $record

In [ ]:
!python evaluate.py --models $BEST

## 7. Download the files to commit

In [ ]:
import shutil
shutil.make_archive("ubridge-ml-artifacts", "zip", "artifacts")
from google.colab import files
files.download("ubridge-ml-artifacts.zip")